## RAG pipe - Data ingestion to vector DB pipeline

In [31]:
import  os

from langchain_classic import text_splitter
from  langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from  langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

### Read all the pdf's inside the directory


In [32]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: Does ChatGPT Know or Does It Guess?.pdf
  ✓ Loaded 4 pages

Processing: redis.pdf
  ✓ Loaded 1 pages

Processing: maven.pdf
  ✓ Loaded 6 pages

Total documents loaded: 11


In [33]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-08-17T07:12:32+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-08-17T07:12:32+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': '../data/pdf_files/Does ChatGPT Know or Does It Guess?.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Does ChatGPT Know or Does It Guess?.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-08-17T07:12:32+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-08-17T07:12:32+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': '../data/pdf_files/Does ChatGPT Know or Does It Guess?.pdf', 'total_pages': 4, 'page': 1, 'page_label': '2', 'source_file': 'Does ChatGPT Know or Does It Guess?.pdf', 'file_type': 'pdf'}, page_conte

## Text Splitting get into chunks


In [34]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [35]:
chunks=split_documents(all_pdf_documents)
chunks

Split 11 documents into 13 chunks

Example chunk:
Content: All commands 
 Run locally 
 docker run -d \ 
 --name redis \ 
 -p 6379:6379 \ 
 redis:latest 
 docker exec -it redis redis-cli 
 -> it executes ‘redis-cli’ command in side ‘redis’ container 
 ●  ping...
Metadata: {'producer': 'Skia/PDF m151', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36', 'creationdate': '2026-08-19T18:20:24+00:00', 'title': 'Redis - Google Docs', 'moddate': '2026-08-19T18:20:24+00:00', 'source': '../data/pdf_files/redis.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'redis.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m151', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36', 'creationdate': '2026-08-19T18:20:24+00:00', 'title': 'Redis - Google Docs', 'moddate': '2026-08-19T18:20:24+00:00', 'source': '../data/pdf_files/redis.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'redis.pdf', 'file_type': 'pdf'}, page_content="All commands \n Run locally \n docker run -d \\ \n --name redis \\ \n -p 6379:6379 \\ \n redis:latest \n docker exec -it redis redis-cli \n -> it executes ‘redis-cli’ command in side ‘redis’ container \n ●  ping \n ○  return a response if redis is alive \n ●  SET name rajesh \n ○  sets a key ‘name’ with value ‘rajesh’ \n ●  GET name \n ○  returns value with key ‘name’ \n ●  DEL name \n ○  deletes key \n ●  EXISTS key \n ○  returns 1 if exists \n ○  return 0 if doesn’t exists \n Distributed locking \n ●  SET invoice:100 app1 NX PX 30000 \n ○  SE

### Embedding and vectorStoreDB

In [36]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

### EmbeddingManager

In [37]:
class EmbeddingManager:
    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):
        """
        Intialise the embedding manager

        Args
            model_name: Huggingface model name for sentece embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        """ Load the sentence transformation model """
        try:
            print(f"Loadig embedding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model load successfully. Embedding dimention: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model : {self.model_name} : {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager=EmbeddingManager()
embedding_manager

Loadig embedding model : all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5990.78it/s]


Model load successfully. Embedding dimention: 384


/var/folders/yb/z0d63kfs0mqg4sq6dtzf7z7c0000gn/T/ipykernel_4168/1365266590.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model load successfully. Embedding dimention: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [38]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 13


### Convert the text to embeddings

In [39]:
texts=[doc.page_content for doc in chunks]
embeddings=embedding_manager.generate_embeddings(texts)
embeddings

Generating embeddings for 13 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.95it/s]

Generated embeddings with shape: (13, 384)


array([[ 0.01662493,  0.02518887, -0.08155739, ...,  0.02424632,
        -0.05494134,  0.05350496],
       [-0.05052933,  0.02609408,  0.00570393, ...,  0.0047403 ,
        -0.01146748,  0.00128122],
       [-0.10945658, -0.031479  ,  0.01543186, ...,  0.06539917,
         0.05613529, -0.01531651],
       ...,
       [-0.00498933, -0.04455218, -0.00337297, ..., -0.03145709,
         0.04378565, -0.08887736],
       [-0.04991696, -0.01273131,  0.04694674, ..., -0.03310116,
         0.01349536, -0.02922702],
       [-0.0645688 , -0.02882269, -0.02315733, ..., -0.02662419,
         0.01466219,  0.06413642]], shape=(13, 384), dtype=float32)

### Store embeddings into VectorStore

In [40]:
vectorstore.add_documents(chunks, embeddings)

Adding 13 documents to vector store...
Successfully added 13 documents to vector store
Total documents in collection: 26


# Retriever pipeline from VectorStore

In [41]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [44]:
rag_retriever.retrieve("Base Model vs. AI Assistant")

Retrieving documents for query: 'Base Model vs. AI Assistant'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.90it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]